In [7]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

batch_size = 64
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

In [8]:
def train_loop(dataloader, model, loss_fn, optimizers):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        for optimizer in optimizers:
            optimizer.step()
            optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [9]:
learning_rate = 1e-3
loss_fn = nn.CrossEntropyLoss()

# adam_params = []
# muon_params = []
# for param in model.parameters():
#     if param.ndim == 2:
#         muon_params.append(param)
#     else:
#         adam_params.append(param)

optimizers = [
    torch.optim.Adam(model.parameters(), lr=learning_rate),
    #torch.optim.Muon(muon_params, lr=learning_rate)
]

epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizers)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.301821  [   64/60000]
loss: 0.550918  [ 6464/60000]
loss: 0.388958  [12864/60000]
loss: 0.485484  [19264/60000]
loss: 0.455032  [25664/60000]
loss: 0.433731  [32064/60000]
loss: 0.382835  [38464/60000]
loss: 0.545517  [44864/60000]
loss: 0.463948  [51264/60000]
loss: 0.507222  [57664/60000]
Test Error: 
 Accuracy: 84.6%, Avg loss: 0.421746 

Epoch 2
-------------------------------
loss: 0.262374  [   64/60000]
loss: 0.338815  [ 6464/60000]
loss: 0.274655  [12864/60000]
loss: 0.390387  [19264/60000]
loss: 0.437647  [25664/60000]
loss: 0.377281  [32064/60000]
loss: 0.309525  [38464/60000]
loss: 0.495507  [44864/60000]
loss: 0.403199  [51264/60000]
loss: 0.449018  [57664/60000]
Test Error: 
 Accuracy: 85.3%, Avg loss: 0.395671 

Epoch 3
-------------------------------
loss: 0.216810  [   64/60000]
loss: 0.309768  [ 6464/60000]
loss: 0.218072  [12864/60000]
loss: 0.334016  [19264/60000]
loss: 0.424255  [25664/60000]
loss: 0.306651  [32064/600